In [1]:
"""
PubMed Abstract Retrieval for Toulmin Argument Framework
Drug: Semaglutide (weight loss claim)
"""

from pymed import PubMed
import time
import json

pubmed = PubMed(tool="ToulminRetrieval", email="put your email")

In [2]:
def _patched_getArticleIds(self, query, max_results):
    parameters = self.parameters.copy()   # base params: tool, email, db=pubmed
    parameters["term"] = query            # the search query string
    parameters["retmax"] = max_results    # how many PMIDs to return
    parameters["sort"] = "relevance"      # rank by relevance, not date
    # Call PubMed esearch endpoint → returns JSON with list of matching PMIDs
    response = self._get(url="/entrez/eutils/esearch.fcgi", parameters=parameters)
    # Extract the list of PMID strings from the response
    article_ids = response.get("esearchresult", {}).get("idlist", [])
    return article_ids

PubMed._getArticleIds = _patched_getArticleIds

In [3]:
queries = {
    "CLAIM_DATA": (
        '(semaglutide[ti]) AND (weight loss OR obesity OR overweight) '
        'AND (efficacy OR effectiveness)'
    ),
    "WARRANT": (
        '(semaglutide[ti]) AND (weight loss) '
        'AND (systematic review OR meta-analysis) AND (adults OR obesity)'
    ),
    "QUALIFIER": (
        '(semaglutide[ti]) AND (weight loss) '
        'AND (long-term OR duration OR maintenance OR discontinuation '
        'OR withdrawal OR weight regain)'
    ),
    "REBUTTAL_PSYCHIATRIC": '(semaglutide[ti]) AND (suicide[ti] OR suicidal ideation or depression or self-injury)'
,
    
    "REBUTTAL_OCULAR": (
        '(semaglutide[ti] OR GLP-1 receptor[ti]) '
        'AND (optic neuropathy OR NAION OR ischemic optic OR vision loss)'
    ),
    "REBUTTAL_GASTROINTESTINAL": (
        '(semaglutide[ti] OR GLP-1 receptor[ti]) '
        'AND (gastrointestinal OR nausea OR vomiting OR pancreatitis '
        'OR gallbladder OR treatment discontinuation) '
        'AND (adverse event OR safety OR side effect)'
    ),
}

In [4]:
def retrieve(query, top_k=5):
    """Retrieve top_k articles from PubMed for a given query."""
    results = []
    for article in pubmed.query(query, max_results=top_k):
        # PMID can sometimes contain newlines; take only the first line
        pmid = str(article.pubmed_id).split("\n")[0].strip() if article.pubmed_id else "N/A"
        title = article.title or "No title"
        abstract = article.abstract or "No abstract"

        # Extract publication types from the raw XML stored by pymed
        # e.g. ["Journal Article", "Randomized Controlled Trial", "Meta-Analysis"]
        pub_types = []
        if hasattr(article, 'xml') and article.xml is not None:
            for pt in article.xml.findall('.//PublicationType'):
                if pt.text:
                    #print(pt.text)
                    pub_types.append(pt.text)

        results.append({
            "pmid": pmid,
            "title": title,
            "abstract": abstract,
            "publication_types": pub_types,
        })
    return results

In [5]:
def run_all(top_k=5, output_file="toulmin_abstracts.json"):
    """Run all queries, merge results, deduplicate, and save to JSON."""
    # Dictionary keyed by PMID to avoid duplicates across queries
    # An article may appear in multiple queries (e.g. a STEP trial
    # could match both CLAIM_DATA and REBUTTAL_GASTROINTESTINAL)
    merged = {}

    for element, query in queries.items():
        print(f"\n--- {element} ---")
        print(f"  Query: {query}")

        results = retrieve(query, top_k=top_k)
        time.sleep(0.5)  # polite rate limiting between queries

        for r in results:
            pmid = r["pmid"]
            if pmid not in merged:
                # First time seeing this PMID: store it with its Toulmin role
                r["toulmin_elements"] = [element]
                merged[pmid] = r
                print(f"  + PMID {pmid}: {r['title']}")
            else:
                # Already seen: just append the additional Toulmin role
                if element not in merged[pmid]["toulmin_elements"]:
                    merged[pmid]["toulmin_elements"].append(element)
                print(f"  = PMID {pmid}: (duplicate, added to {element})")

    # Convert to list and save
    output = list(merged.values())
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    print(f"\nDone. {len(output)} unique articles saved to {output_file}")


if __name__ == "__main__":
    run_all(top_k=5, output_file="toulmin_abstracts.json")





--- CLAIM_DATA ---
  Query: (semaglutide[ti]) AND (weight loss OR obesity OR overweight) AND (efficacy OR effectiveness)
  + PMID 36578889: Efficacy and Safety of Semaglutide for Weight Loss in Obesity Without Diabetes: A Systematic Review and Meta-Analysis.
  + PMID 33567185: Once-Weekly Semaglutide in Adults with Overweight or Obesity.
  + PMID 33755728: Effect of Continued Weekly Subcutaneous Semaglutide vs Placebo on Weight Loss Maintenance in Adults With Overweight or Obesity: The STEP 4 Randomized Clinical Trial.
  + PMID 33667417: Semaglutide 2·4 mg once a week in adults with overweight or obesity, and type 2 diabetes (STEP 2): a randomised, double-blind, double-dummy, placebo-controlled, phase 3 trial.
  + PMID 36216945: Two-year effects of semaglutide in adults with overweight or obesity: the STEP 5 trial.

--- WARRANT ---
  Query: (semaglutide[ti]) AND (weight loss) AND (systematic review OR meta-analysis) AND (adults OR obesity)
  = PMID 36578889: (duplicate, added to WARRA